# Exam 2021, 8.00-13.00 for the course 1MS041 (Introduction to Data Science / Introduktion till dataanalys)

## Instructions:
1. Complete the problems by following instructions.
2. When done, submit this file with your solutions saved, following the instruction sheet.

This exam has 3 problems for a total of 40 points, to pass you need
20 points.

## Some general hints and information:
* Try to answer all questions even if you are uncertain.
* Comment your code, so that if you get the wrong answer I can understand how you thought
this can give you some points even though the code does not run.
* Follow the instruction sheet rigorously.
* This exam is partially autograded, but your code and your free text answers are manually graded anonymously.
* If there are any questions, please ask the exam guards, they will escalate it to me if necessary.
* I (Benny) will visit the exam room at around 10:30 to see if there are any questions.

## Tips for free text answers
* Be VERY clear with your reasoning, there should be zero ambiguity in what you are referring to.
* If you want to include math, you can write LaTeX in the Markdown cells, for instance `$f(x)=x^2$` will be rendered as $f(x)=x^2$ and `$$f(x) = x^2$$` will become an equation line, as follows
$$f(x) = x^2$$
Another example is `$$f_{Y \mid X}(y,x) = P(Y = y \mid X = x) = \exp(\alpha \cdot x + \beta)$$` which renders as
$$f_{Y \mid X}(y,x) = P(Y = y \mid X = x) = \exp(\alpha \cdot x + \beta)$$

## Finally some rules:
* You may not communicate with others during the exam, for example:
    * You cannot ask for help in Stack-Overflow or other such help forums during the Exam.
    * You may not communicate with AI's, for instance ChatGPT.
    * Your on-line and off-line activity is being monitored according to the examination rules.

## Good luck!

In [8]:
# Insert your anonymous exam ID as a string in the variable below
examID="XXX"


---
## Exam vB, PROBLEM 1
Maximum Points = 8


## Probability warmup
Let's say we have an exam question which consists of $20$ yes/no questions. 
From past performance of similar students, a randomly chosen student will know the correct answer to $N \sim \text{binom}(20,11/20)$ questions. Furthermore, we assume that the student will guess the answer with equal probability to each question they don't know the answer to, i.e. given $N$ we define $Z \sim \text{binom}(20-N,1/2)$ as the number of correctly guessed answers. Define $Y = N + Z$, i.e., $Y$ represents the number of total correct answers.

We are interested in setting a deterministic threshold $T$, i.e., we would pass a student at threshold $T$ if $Y \geq T$. Here $T \in \{0,1,2,\ldots,20\}$.

1. [5p] For each threshold $T$, compute the probability that the student *knows* less than $10$ correct answers given that the student passed, i.e., $N < 10$. Put the answer in `problem11_probabilities` as a list.
2. [3p] What is the smallest value of $T$ such that if $Y \geq T$ then we are 90\% certain that $N \geq 10$?

In [9]:

# Hint the PMF of N is p_N(k) where p_N is
p = 11/20
from scipy.special import binom as binomial
p_N = lambda k: binomial(20,k)*(1-p)^(20-k)*(p)^k

In [1]:

# Hint the PMF of N is p_N(k) where p_N is
p = 11/20
from scipy.special import binom as binomial

# Fixed syntax error (use ** for power)
p_N = lambda k: binomial(20,k) * ((1-p)**(20-k)) * (p**k)
# PMF for Z given N: Binom(20-N, 0.5)
p_Z_given_N = lambda z, n: binomial(20-n, z) * (0.5**(20-n))

# Joint PMF P(N=n, Y=y). Note Y = N + Z, so Z = Y - N.
def joint_p(n, y):
    z = y - n
    if z < 0 or z > (20-n):
        return 0
    return p_N(n) * p_Z_given_N(z, n)

# Compute P(N < 10 | Y >= T)
problem10_probabilities = []
for t in range(21):
    # Numerator: Sum joint prob where N < 10 AND Y >= t
    num = 0
    for n in range(10): # n = 0 to 9
        for y in range(t, 21):
            num += joint_p(n, y)
    
    # Denominator: Sum joint prob where Y >= t (marginal P(Y >= t))
    denom = 0
    for n in range(21):
        for y in range(t, 21):
            denom += joint_p(n, y)
            
    if denom == 0:
        problem10_probabilities.append(0)
    else:
        problem10_probabilities.append(num / denom)


In [ ]:
# Part 2: What is the smallest T such that P(N >= 10 | Y >= T) >= 0.9?
# This is equivalent to P(N < 10 | Y >= T) <= 0.1
problem12_T = -1
for t in range(21):
    if problem10_probabilities[t] <= 0.1:
        problem12_T = t
        break

# problem12_T should be the integer found
problem12_T = XXX


---
## Exam vB, PROBLEM 2
Maximum Points = 8


## Random variable generation and transformation

The purpose of this problem is to show that you can implement your own sampler, this will be built in the following three steps:

1. [2p] Implement a Linear Congruential Generator where you tested out a good combination (a large $M$ with $a,b$ satisfying the Hull-Dobell (Thm 6.8)) of parameters. Follow the instructions in the code block.
2. [2p] Using a generator construct random numbers from the uniform $[0,1]$ distribution.
3. [4p] Using a uniform $[0,1]$ random generator, generate samples from 

$$p_0(x) = \frac{\pi}{2}|\sin(2\pi x)|, \quad x \in [0,1] \enspace .$$

Using the **Accept-Reject** sampler (**Algorithm 1** in TFDS notes) with sampling density given by the uniform $[0,1]$ distribution.

In [ ]:
def problem2_LCG(size=None, seed = 0):
    """
    A linear congruential generator that generates pseudo random numbers according to size.
    """
    # Using parameters from Numerical Recipes (or similar high-quality LCG)
    m = 2**32
    a = 1664525
    c = 1013904223
    
    out = []
    x = seed
    for _ in range(size):
        x = (a * x + c) % m
        out.append(x)
    
    return out

In [ ]:

def problem2_uniform(generator=None, period = 1, size=None, seed=0):
    """
    Takes a generator and produces samples from the uniform [0,1] distribution according
    to size.
    
    Parameters
    -------------
    generator : a function of type generator(size,seed) and produces the same result as problem2_LCG, i.e. pseudo random numbers in the range {0,1,...,period-1}
    period : the period of the generator
    seed : the seed to be used in the generator provided
    size : an integer denoting how many samples should be produced
    
    Returns
    --------------
    out : a list of the uniform pseudo random numbers
    """
    """
    Takes a generator and produces samples from the uniform [0,1] distribution.
    """
   
    """
    Takes a generator and produces samples from the uniform [0,1] distribution.
    """
    # Generate raw integers
    raw_values = generator(size=size, seed=seed)
    
    # Normalize by the period (m) to get [0,1]
    out = [val / period for val in raw_values]
    
    return out
    
 
    


In [ ]:

def problem2_accept_reject(uniformGenerator=None, size=None, seed=0):
    """
    Takes a generator that produces uniform pseudo random [0,1] numbers 
    and produces samples from (pi/2)*abs(sin(x*2*pi)) using an Accept-Reject
    sampler with the uniform distribution as the proposal distribution
    
    Parameters
    -------------
    generator : a function of the type generator(size,seed) that produces uniform pseudo random
    numbers from [0,1]
    seed : the seed to be used in the generator provided
    size : an integer denoting how many samples should be produced
    
    Returns
    --------------
    out : a list of the pseudo random numbers with the specified distribution
    """
    
    out = []
    current_seed = seed
    
    # The max of |sin| is 1. The normalizing constant of the target is pi/2.
    # f(x) = (pi/2)|sin(2*pi*x)|.
    # Proposal g(x) = 1 (Uniform).
    # We need M such that f(x) <= M * g(x). Max f(x) is pi/2.
    # So set M = pi/2.
    # Acceptance condition: U <= f(Y) / (M*g(Y))
    # U <= ((pi/2)|sin(2*pi*Y)|) / (pi/2 * 1)
    # U <= |sin(2*pi*Y)|
    
    while len(out) < size:
        # We need two uniform numbers per attempt (Y for proposal, U for acceptance)
        # To ensure distinct randomness, we increment seed or rely on state management. 
        # Here we just request 2 numbers at a time with updating seeds.
        rands = uniformGenerator(size=2, seed=current_seed)
        current_seed += 2 
        
        y = rands[0] # Candidate from uniform
        u = rands[1] # Acceptance variable
        
        if u <= np.abs(np.sin(2 * np.pi * y)):
            out.append(y)
    
    return out
    


---
#### Local Test for Exam vB, PROBLEM 2
Evaluate cell below to make sure your answer is valid.                             You **should not** modify anything in the cell below when evaluating it to do a local test of                             your solution.
You may need to include and evaluate code snippets from lecture notebooks in cells above to make the local test work correctly sometimes (see error messages for clues). This is meant to help you become efficient at recalling materials covered in lectures that relate to this problem. Such local tests will generally not be available in the exam.

In [ ]:

# If you managed to solve all three parts you can test the following code to see if it runs
# you have to change the period to match your LCG though, this is marked as XXX.
# It is a very good idea to check these things using the histogram function in sagemath
# try with a larger number of samples, up to 10000 should run

print("LCG output: %s" % problem2_LCG(size=10, seed = 1))

period = XXX

print("Uniform sampler %s" % problem2_uniform(generator=problem2_LCG, period = period, size=10, seed=1))

uniform_sampler = lambda size,seed: problem2_uniform(generator=problem2_LCG, period = period, size=size, seed=seed)

print("Accept-Reject sampler %s" % problem2_accept_reject(uniformGenerator = uniform_sampler,n_iterations=20,seed=1))

In [ ]:

# If however you did not manage to implement either part 1 or part 2 but still want to check part 3, you can run the code below

def testUniformGenerator(size,seed):
    set_random_seed(seed)
    
    return [random() for s in range(size)]

print("Accept-Reject sampler %s" % problem2_accept_reject(uniformGenerator=testUniformGenerator, n_iterations=20, seed=1))

---
## Exam vB, PROBLEM 3
Maximum Points = 8


## Concentration of measure

As you recall, we said that concentration of measure was simply the phenomenon where we expect that the probability of a large deviation of some quantity becoming smaller as we observe more samples: [0.4 points per correct answer]

1. Which of the following will exponentially concentrate, i.e. for some $C_1,C_2,C_3,C_4 $ 
$$
    P(Z - \mathbb{E}[Z] \geq \epsilon) \leq C_1 e^{-C_2 n \epsilon^2} \wedge C_3 e^{-C_4 n (\epsilon+1)} \enspace .
$$

    1. The empirical mean of i.i.d. sub-Gaussian random variables?
    2. The empirical mean of i.i.d. sub-Exponential random variables?
    3. The empirical mean of i.i.d. random variables with finite variance?
    4. The empirical variance of i.i.d. random variables with finite variance?
    5. The empirical variance of i.i.d. sub-Gaussian random variables?
    6. The empirical variance of i.i.d. sub-Exponential random variables?
    7. The empirical third moment of i.i.d. sub-Gaussian random variables?
    8. The empirical fourth moment of i.i.d. sub-Gaussian random variables?
    9. The empirical mean of i.i.d. deterministic random variables?
    10. The empirical tenth moment of i.i.d. Bernoulli random variables?

2. Which of the above will concentrate in the weaker sense, that for some $C_1$
$$
    P(Z - \mathbb{E}[Z] \geq \epsilon) \leq \frac{C_1}{n \epsilon^2}?
$$

In [ ]:

# Answers to part 1: Exponential concentration
# 1 (Mean Sub-G), 2 (Mean Sub-E), 5 (Var Sub-G ~ Mean Sub-E), 6 (Var Sub-E), 
# 7 (3rd Moment Sub-G), 8 (4th Moment Sub-G), 9 (Deterministic), 10 (Bernoulli Bounded)
problem3_answer_1 = [1, 2, 5, 6, 7, 8, 9, 10]

In [ ]:

# Answers to part 2, which of the alternatives concentrate in the weaker sense, answer as a list
# i.e. [1,4,5] that is example 1, 4, and 5 concentrate
# Answers to part 2: Weaker concentration (Chebyshev)
problem3_answer_2 = [3, 4]

---
## Exam vB, PROBLEM 4
Maximum Points = 8


## SMS spam filtering [8p]

In the following problem we will explore SMS spam texts. The dataset is the `SMS Spam Collection Dataset` and we have provided for you a way to load the data. If you run the appropriate cell below, the result will be in the `spam_no_spam` variable. The result is a `list` of `tuples` with the first position in the tuple being the SMS text and the second being a flag `0 = not spam` and `1 = spam`.

1. [3p] Let $X$ be the random variable that represents each SMS text (an entry in the list), and let $Y$ represent whether text is spam or not i.e. $Y \in \{0,1\}$. Thus $\mathbb{P}(Y = 1)$ is the probability that we get a spam. The goal is to estimate:
$$
    \mathbb{P}(Y = 1 | \text{"free" or "prize" is in } X) \enspace .
$$
That is, the probability that the SMS is spam given that "free" or "prize" occurs in the SMS. 
Hint: it is good to remove the upper/lower case of words so that we can also find "Free" and "Prize"; this can be done with `text.lower()` if `text` a string.

2. [3p] Provide a "90\%" interval of confidence around the true probability. I.e. use the Hoeffding inequality to obtain for your estimate $\hat P$ of the above quantity. Find $l > 0$ such that the following holds:
$$
    \mathbb{P}(\hat P - l \leq \mathbb{E}[\hat P] \leq \hat P + l) \geq 0.9 \enspace .
$$
3. [2p] Repeat the two exercises above for "free" appearing twice in the SMS.

In [16]:
import pandas as pd
# This URL points to a reliable mirror of the standard UCI SMS dataset
url = 'https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv'

# Read the tab-separated file
# We specify header=None because the file doesn't have column names
# We name the columns 'label' and 'message' manually
spam_no_spam = pd.read_table(url, header=None, names=['label', 'message'])

# Display the first few rows to verify
# Run this cell to get the SMS text data




In [17]:
import re
spam_no_spam['label'] = spam_no_spam['label'].map({'spam': 1, 'ham': 0})
#print(spam_no_spam[:10])
spam_no_spam["message"] = spam_no_spam['message'].str.lower()
#print(spam_no_spam[:10])

spam_no_spam['guess'] = 0
#print(spam_no_spam[:10])

spam_no_spam['guess'] = spam_no_spam['message'].str.contains('free|prize').astype(int)
#print(spam_no_spam[:10])

col1_1 = ((spam_no_spam['label'] == 1) & (spam_no_spam['guess'] == 0)).sum()
col1_10 = ((spam_no_spam['label'] == 1) & (spam_no_spam['guess'] == 1)).sum()


# fill in the calculated l from part 2 he
# fill in the estimate for part 1 here (should be a number between 0 and 1)
denominator = (spam_no_spam['guess'] == 1).sum()
problem4_hatP = col1_10/denominator
print(problem4_hatP)

0.808695652173913


In [18]:
import numpy as np
def epsilon_bernoulli(n,alpha):
    return np.sqrt(-1/(2*n)*np.log((alpha)/2))

Z_obs = spam_no_spam['guess'].tolist()
Y_obs = spam_no_spam['label'].tolist()
Y_mid_Z1 = [y for z,y in zip(Z_obs,Y_obs) if z == 1]
# fill in the calculated l from part 2 here
epsilon = epsilon_bernoulli(len(Y_mid_Z1),0.05)
problem4_l = (problem4_hatP - epsilon, problem4_hatP + epsilon)
print(problem4_l)

(np.float64(0.7355779244506618), np.float64(0.8818133798971642))


In [19]:
import pandas as pd
# This URL points to a reliable mirror of the standard UCI SMS dataset
url = 'https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv'

# Read the tab-separated file
# We specify header=None because the file doesn't have column names
# We name the columns 'label' and 'message' manually
spam_no_spam = pd.read_table(url, header=None, names=['label', 'message'])

# Display the first few rows to verify
# Run this cell to get the SMS text data

spam_no_spam['label'] = spam_no_spam['label'].map({'spam': 1, 'ham': 0})
#print(spam_no_spam[:10])
spam_no_spam["message"] = spam_no_spam['message'].str.lower()
#print(spam_no_spam[:10])

spam_no_spam['guess'] = 0
#print(spam_no_spam[:10])

spam_no_spam['guess'] = (
    spam_no_spam['message'].str.lower().str.count(r'\bfree\b') >= 2
).astype(int)
print(spam_no_spam[:10])

col1_1 = ((spam_no_spam['label'] == 1) & (spam_no_spam['guess'] == 0)).sum()
col1_10 = ((spam_no_spam['label'] == 1) & (spam_no_spam['guess'] == 1)).sum()


# fill in the calculated l from part 2 he
# fill in the estimate for part 1 here (should be a number between 0 and 1)
denominator = (spam_no_spam['guess'] == 1).sum()
problem4_hatP2 = col1_10/denominator
print(problem4_hatP2)


# fill in the estimate for hatP for the double free question in part 3 here (should be a number between 0 and 1)
problem4_hatP2 = problem4_hatP2

   label                                            message  guess
0      0  go until jurong point, crazy.. available only ...      0
1      0                      ok lar... joking wif u oni...      0
2      1  free entry in 2 a wkly comp to win fa cup fina...      0
3      0  u dun say so early hor... u c already then say...      0
4      0  nah i don't think he goes to usf, he lives aro...      0
5      1  freemsg hey there darling it's been 3 week's n...      0
6      0  even my brother is not like to speak with me. ...      0
7      0  as per your request 'melle melle (oru minnamin...      0
8      1  winner!! as a valued network customer you have...      0
9      1  had your mobile 11 months or more? u r entitle...      1
0.9767441860465116


In [20]:
Z_obs = spam_no_spam['guess'].tolist()
Y_obs = spam_no_spam['label'].tolist()
Y_mid_Z1 = [y for z,y in zip(Z_obs,Y_obs) if z == 1]
# fill in the calculated l from part 2 here
epsilon = epsilon_bernoulli(len(Y_mid_Z1),0.05)
problem4_l2 = (problem4_hatP2 - epsilon, problem4_hatP2 + epsilon)
print(problem4_l)
# fill in the estimate for l for the double free question in part 3 here
problem4_l2 = (problem4_hatP2 - epsilon, 1)
print(problem4_l2)

(np.float64(0.7355779244506618), np.float64(0.8818133798971642))
(np.float64(0.7696356465295238), 1)


---
## Exam vB, PROBLEM 5
Maximum Points = 8


## Markovian travel

The dataset `Travel Dataset - Datathon 2019` is a simulated dataset designed to mimic real corporate travel systems -- focusing on flights and hotels. The file is at `data/flights.csv` in the same folder as `Exam.ipynb`, i.e. you can use the path `data/flights.csv` from the notebook to access the file.

1. [2p] In the first code-box 
    1. Load the csv from file `data/flights.csv`
    2. Fill in the value of the variables as specified by their names.
2. [2p] In the second code-box your goal is to estimate a Markov chain transition matrix for the travels of these users. For example, if we enumerate the cities according to alphabetical order, the first city `'Aracaju (SE)'` would correspond to $0$. Each row of the file corresponds to one flight, i.e. it has a starting city and an ending city. We model this as a stationary Markov chain, i.e. each user's travel trajectory is a realization of the Markov chain, $X_t$. Here, $X_t$ is the current city the user is at, at step $t$, and $X_{t+1}$ is the city the user travels to at the next time step. This means that to each row in the file there is a corresponding pair $(X_{t},X_{t+1})$. The stationarity assumption gives that for all $t$ there is a transition density $p$ such that $P(X_{t+1} = y | X_t = x) = p(x,y)$ (for all $x,y$). The transition matrix should be `n_cities` x `n_citites` in size.
3. [2p] Use the transition matrix to compute out the stationary distribution.
4. [2p] Given that we start in 'Aracaju (SE)' what is the probability that after 3 steps we will be back in 'Aracaju (SE)'?

In [21]:
import numpy as np
import pandas as pd
d = pd.read_csv('data/flights.csv', sep=',')
print(list(d.columns.values))

#print(d['from'].value_counts()) This gives all distinct values and their counts
#print(d['to'].value_counts()) ^^^

#d['userCode'].nunique() #This gives all unique userCodes

#print(d.info)   #This gives the whole table but and a count of number of rows.
#print(len(d))  #This gives the number of observations (rows) in the data frame.


number_of_cities = 9
number_of_userCodes = 1335
number_of_observations = 271888

['travelCode', 'userCode', 'from', 'to', 'flightType', 'price', 'time', 'distance', 'agency', 'date']


In [22]:

# This is a very useful function that you can use for part 2. You have seen this before when parsing the
# pride and prejudice book.

def makeFreqDict(myDataList):
    '''Make a frequency mapping out of a list of data.

    Param myDataList, a list of data.
    Return a dictionary mapping each unique data value to its frequency count.'''

    freqDict = {} # start with an empty dictionary

    for res in myDataList:
        if res in freqDict: # the data value already exists as a key
                freqDict[res] = freqDict[res] + 1 # add 1 to the count using sage integers
        else: # the data value does not exist as a key value
            freqDict[res] = 1 # add a new key-value pair for this new data value, frequency 1

    return freqDict # return the dictionary created

In [ ]:
d1 = d.copy()
cities = makeFreqDict(d1["from"])
#print(cities)
unique_cities = sorted(set(cities)) # The unique cities
#print(unique_cities)
n_cities = len(unique_cities) # The number of unique citites
#print(n_cities)

# Count the different transitions

 # A list containing tuples ex: ('Aracaju (SE)','Rio de Janeiro (RJ)') of all transitions in the text
to_from_list = []
for index, row in d1.iterrows():
    to_from_list.append((row['from'], row['to']))

transitions = to_from_list

transition_counts = makeFreqDict(transitions) # A dictionary that counts the number of each transition 
# ex: ('Aracaju (SE)','Rio de Janeiro (RJ)'):4

indexToCity = {index: value for index, value in enumerate(unique_cities)} # A dictionary that maps the n-1 number to the n:th unique_city,
print(indexToCity)
# ex: 0:'Aracaju (SE)'


cityToIndex = {city: idx for idx, city in enumerate(unique_cities)}# The inverse function of indexToWord, 
print(cityToIndex)
# ex: 'Aracaju (SE)':0
# Part 3, finding the maximum likelihood estimate of the transition matrix

#transition_matrix =  # a numpy array of size (n_cities,n_cities)

# The transition matrix should be ordered in such a way that
# p_{'Aracaju (SE)','Rio de Janeiro (RJ)'} = transition_matrix[cityToIndex['Aracaju (SE)'],cityToIndex['Rio de Janeiro (RJ)']]
# and represents the probability of travelling Aracaju (SE)->Rio de Janeiro (RJ)

# Make sure that the transition_matrix does not contain np.nan from division by zero for instance

# 1. Initialize a zero matrix of size (n_cities, n_cities)
transition_counts = np.zeros((n_cities, n_cities))

# 2. Count the transitions
# We iterate through the 'from' and 'to' columns simultaneously
for origin, destination in zip(d1['from'], d1['to']):
    i = cityToIndex[origin]      # Get row index
    j = cityToIndex[destination] # Get column index
    transition_counts[i, j] += 1

# 3. Calculate the MLE (Normalize row-wise)
# Sum each row to get the total number of flights departing from each city
row_sums = transition_counts.sum(axis=1, keepdims=True)

# Divide counts by row sums to get probabilities. 
# 'where=row_sums!=0' prevents division by zero (resulting in NaNs).
# If a row sums to 0, it remains 0.
transition_matrix = np.divide(
    transition_counts, 
    row_sums, 
    out=np.zeros_like(transition_counts), 
    where=row_sums!=0
)

print(transition_matrix)

{0: 'Aracaju (SE)', 1: 'Brasilia (DF)', 2: 'Campo Grande (MS)', 3: 'Florianopolis (SC)', 4: 'Natal (RN)', 5: 'Recife (PE)', 6: 'Rio de Janeiro (RJ)', 7: 'Salvador (BH)', 8: 'Sao Paulo (SP)'}
{'Aracaju (SE)': 0, 'Brasilia (DF)': 1, 'Campo Grande (MS)': 2, 'Florianopolis (SC)': 3, 'Natal (RN)': 4, 'Recife (PE)': 5, 'Rio de Janeiro (RJ)': 6, 'Salvador (BH)': 7, 'Sao Paulo (SP)': 8}
[[0.         0.12983559 0.14487965 0.23218891 0.1057651  0.13120567
  0.07570385 0.07839029 0.10203095]
 [0.15702265 0.         0.14675591 0.25273726 0.09704669 0.12417557
  0.06478443 0.06527178 0.09220572]
 [0.15520318 0.12999309 0.         0.23751007 0.1019627  0.12979164
  0.0695004  0.07252216 0.10351675]
 [0.15079296 0.1357189  0.14398869 0.         0.11705079 0.13275294
  0.10131375 0.10119162 0.11719036]
 [0.16544797 0.1255253  0.14889057 0.28193814 0.         0.12161708
  0.03992268 0.0389141  0.07774416]
 [0.16023622 0.1253937  0.14796588 0.24963911 0.09494751 0.
  0.06223753 0.06404199 0.09553806]
 [

In [28]:

# This should be a numpy array of length n_cities which sums to 1 and is all positive
# 1. Calculate eigenvalues and eigenvectors of the Transpose
# We use P.T because we want the left eigenvector: v P = v
eigenvalues, eigenvectors = np.linalg.eig(transition_matrix.T)

# 2. Find the index of the eigenvalue closest to 1
# (Theoretical Markov chains always have an eigenvalue of 1)
idx = np.argmin(np.abs(eigenvalues - 1))

# 3. Extract the corresponding eigenvector
stationary_dist = np.real(eigenvectors[:, idx])

# 4. Normalize it so the probabilities sum to 1
stationary_dist = stationary_dist / stationary_dist.sum()

print("Stationary Distribution:")
print(stationary_dist)

stationary_distribution_problem5 = stationary_dist

Stationary Distribution:
[0.13690932 0.1132047  0.12780262 0.21081107 0.08752133 0.11210498
 0.06184532 0.06290826 0.0868924 ]


In [34]:

# Compute the return probability for part 3 of problem 5
"""4.Given that we start in 'Aracaju (SE)' what is the probability that after 3 steps we will be back in 'Aracaju (SE)'?"""
##Aracaju is number 0 in indexToCity
state_index = cityToIndex['Aracaju (SE)']  
state1 = transition_matrix[state_index, :]  # After 1 step
state2 = np.dot(state1, transition_matrix)  # After 2 steps
state3 = np.dot(state2, transition_matrix)  # After 3 steps

return_probability_problem5 = state3[state_index]
print("Return probability after 3 steps starting from 'Aracaju (SE)':", return_probability_problem5)

Return probability after 3 steps starting from 'Aracaju (SE)': 0.13331717737273133


---
#### Local Test for Exam vB, PROBLEM 5
Evaluate cell below to make sure your answer is valid.                             You **should not** modify anything in the cell below when evaluating it to do a local test of                             your solution.
You may need to include and evaluate code snippets from lecture notebooks in cells above to make the local test work correctly sometimes (see error messages for clues). This is meant to help you become efficient at recalling materials covered in lectures that relate to this problem. Such local tests will generally not be available in the exam.

In [35]:
# Once you have created all your functions, you can make a small test here to see
# what would be generated from your model.
import numpy as np

start = np.zeros(shape=(n_cities,1))
start[cityToIndex['Aracaju (SE)'],0] = 1

current_pos = start
for i in range(10):
    random_word_index = np.random.choice(range(n_cities),p=current_pos.reshape(-1))
    current_pos = np.zeros_like(start)
    current_pos[random_word_index] = 1
    print(indexToCity[random_word_index],end='->')
    current_pos = (current_pos.T@transition_matrix).T

Aracaju (SE)->Sao Paulo (SP)->Brasilia (DF)->Campo Grande (MS)->Natal (RN)->Salvador (BH)->Campo Grande (MS)->Natal (RN)->Florianopolis (SC)->Sao Paulo (SP)->

---
## Exam vB, PROBLEM 6
Maximum Points = 8


## Black box testing

In the following problem we will continue with our SMS spam / nospam data. This time we will try to approach the problem as a pattern recognition problem. For this particular problem I have provided you with everything -- data is prepared, split into train-test sets and a black-box model has been fitted on the training data and predicted on the test data. Your goal is to calculate test metrics and provide guarantees for each metric.

1. [2p] Compute precision for class 1 (see notes 8.3.2 for definition), then provide an interval using Hoeffding's inequality for a 95\% confidence.
2. [2p] Compute recall for class 1(see notes 8.3.2 for definition), then provide an interval using Hoeffding's inequality for a 95\% interval.
3. [2p] Compute accuracy (0-1 loss), then provide an interval using Hoeffding's inequality for a 95\% interval.
4. [2p] If we would have used a classifier with VC-dimension 3, would we have obtained a smaller interval for accuracy by using all data?

In [ ]:

# The code below will load data, split the data into train and test and run a "black box" algorithm on it
# the result of the "black box" is stored in predictions_problem6, the true values will be stored in
# Y_test_problem6
import exam_extras
from exam_extras import load_sms_problem6
X_problem6, Y_problem6 = load_sms_problem6()
pred = np.array(predictions_problem6)
truth = np.array(Y_test_problem6)

# TP: Predicted 1 and True 1
tp = ((pred == 1) & (truth == 1)).sum()
# Predicted Positives: Predicted 1
pred_pos = (pred == 1).sum()

problem6_precision = tp / pred_pos

In [ ]:

# Compute the precision of predictions_problem6 with respect to Y_test_problem6
n_prec = (pred == 1).sum()
epsilon_prec = np.sqrt((1/(2*n_prec)) * np.log(2/0.05))
problem6_precision_l = epsilon_prec
problem6_precision = XXX

In [ ]:

# Compute the interval length l of precision of predictions_problem6 with respect to Y_test_problem6, with the same definition of l as in problem 4
# Recall = TP / (TP + FN) = P(Pred=1 | Y=1)
# n = number of actual positives
# Recall = TP / (TP + FN) = P(Pred=1 | Y=1)
# n = number of actual positives
true_pos = (truth == 1).sum()
tp = ((pred == 1) & (truth == 1)).sum()

problem6_recall = tp / true_pos

In [ ]:

# Repeat the same procedure but for recall
# Hoeffding for recall (n = number of actual positives)
epsilon_rec = np.sqrt((1/(2*true_pos)) * np.log(2/0.05))
problem6_recall_l = epsilon_rec

In [ ]:
# Hoeffding for recall (n = number of actual positives)
epsilon_rec = np.sqrt((1/(2*true_pos)) * np.log(2/0.05))
problem6_recall_l = epsilon_rec

In [ ]:

# Repeat the same procedure but for accuracy or 0-1 loss
# Accuracy = (TP + TN) / Total
problem6_accuracy = (pred == truth).mean()


In [ ]:
# Hoeffding for accuracy (n = total samples)
n_total = len(truth)
epsilon_acc = np.sqrt((1/(2*n_total)) * np.log(2/0.05))
problem6_accuracy_l = epsilon_acc


In [ ]:

# Below you will calculate the interval parameter l for a classifier running on all data with a VC dimension of 3
# put the value in problem6_VC_l and answer problem_VC_smaller as True if the interval is smaller than the test-accuracy above
# if not answer False. Make sure you replace XXX with something even if you only answer one of them.
problem6_VC_l = XXX # number
problem6_VC_smaller = XXX #True / False


# VC Dimension interval using all data
# Usually approximated by sqrt( (d*ln(2N/d) + ln(4/delta)) / N ) or similar standard bound provided in course.
# Using a common standard form for VC bound (Vapnik):
# epsilon = sqrt( (8/N) * ( ln(4/delta) + d_vc * ln(2eN/d_vc) ) )
# Note: The question asks "would we have obtained a smaller interval by using ALL data?"
# So N should be X_problem6.shape[0] (Total data), not just test data.

d_vc = 3
delta = 0.05
N_all = X_problem6.shape[0] # Total dataset size

# Implementation of a standard VC bound (adjust formula if course notes specified a simplified version)
import math
problem6_VC_l = math.sqrt( (8/N_all) * (math.log(4/delta) + d_vc * math.log((2 * math.e * N_all) / d_vc)) )

# Compare with the test-set only accuracy interval (problem6_accuracy_l)
# If the VC bound using all data is smaller than the Hoeffding bound on test data:
problem6_VC_smaller = problem6_VC_l < problem6_accuracy_l